# 02 · Feature engineering
Two monthly panels are built from the cleaned events:

* **Conflict level** – 4 cases × 153 months (Jan 2014 – Sep 2026).
* **Sub-national level** – 222 (case, country, first-level area) series × 153 months, zero-filled.

Every model feature is computed so that it is **known at the end of the forecast-origin month** (no look-ahead),
and annual context (displacement, World Bank) enters with a **one-year lag**.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
from src import config as C, viz

In [2]:
from src import features
ft = features.run()
cm, am = ft["case_month"], ft["admin1_month"]
print("conflict-month panel", cm.shape, "| sub-national panel", am.shape)
cm[cm.conflict_case == "US–Iran"].tail(6)[["month", "deaths", "n_events", "n_admin1", "hhi", "outside_core_share",
                                         "civ_share", "uncert_index", "n_outlets", "y_rm3", "momentum_3v12"]]

conflict-month panel (612, 79) | sub-national panel (33966, 75)


,month,deaths,n_events,n_admin1,hhi,outside_core_share,civ_share,uncert_index,n_outlets,y_rm3,momentum_3v12
606,2026-04-01,590.306,62.000,28.000,0.322,0.033,0.450,0.118,24.000,6.786,3.329
607,2026-05-01,24.214,3.000,4.000,0.401,0.330,0.422,0.022,4.000,5.852,2.460
608,2026-06-01,11.688,4.000,4.000,0.384,0.086,0.316,0.000,2.000,4.050,1.039
609,2026-07-01,72.669,32.000,13.000,0.204,0.069,0.339,0.083,8.000,3.356,0.078
610,2026-08-01,4.000,2.000,2.000,0.500,0.000,0.500,0.250,2.000,2.817,-0.310
611,2026-09-01,22.000,7.000,5.000,0.306,0.000,0.273,0.000,6.000,3.015,-0.008


## Feature dictionary

In [3]:
pd.DataFrame(list(features.FEATURE_DICTIONARY.items()), columns=["feature", "meaning"])

,feature,meaning
0,y_l0..y_l12,log1p(deaths) at the origin month and its lags...
1,y_rm3/6/12,rolling mean of log deaths over 3/6/12 months ...
2,y_rstd6,6-month volatility of log deaths
3,y_max12,peak log deaths in the last year (memory of pa...
4,y_ewm,exponentially weighted log deaths (alpha 0.5)
5,momentum_3v12,short-term vs long-term intensity (positive = ...
6,accel,month-on-month change in log deaths
7,zero_streak,consecutive months with no deaths (dormancy)
8,cum_active_months,number of months with deaths so far (conflict ...
9,log_events_rm3,"log of events per month, 3-month mean (operati..."


### Feature families and the military/analytic idea behind each
| Family | Examples | Idea |
|---|---|---|
| Intensity memory | lags, rolling means, EWM, 12-month peak | Wars have inertia; the best predictor of next month is a smoothed recent level |
| Momentum | `momentum_3v12`, `accel` | Escalation shows up as short-term intensity running above the long-term norm |
| Tempo | events/month, references/month | Operational tempo (number of engagements) vs lethality |
| Fog of war | `uncert_index`, `agg_share`, `lowprec_share` | How much the reporting picture is degraded |
| Character of violence | civilian share, state/one-sided share, focal share | Shift from battlefield to punitive violence |
| Geography | active areas, HHI concentration, share outside core theatre | Front widening and horizontal escalation |
| Seasonality | target-month sine/cosine, same-month-last-year | Campaigning seasons, weather |
| Context (lag 1y) | IDPs, refugees, GDP growth, inflation | Structural stress |
| Hierarchy (sub-national) | conflict-level intensity, area's share of the conflict | Areas inherit their war's tempo |

In [4]:
from src.features import make_supervised
sup = make_supervised(cm, ["conflict_case"], horizons=(1,))
num = ["target_y", "y_l0", "y_rm3", "y_rm12", "momentum_3v12", "log_events_rm3", "log_sources_rm3", "civ_share_rm3",
       "uncert_index_rm3", "n_admin1_rm3", "hhi_rm3", "outside_core_share_rm3", "zero_streak", "y_seasonal"]
corr = sup[num].corr(method="spearman")["target_y"].drop("target_y").sort_values()
corr.rename("Spearman correlation with next month's log deaths").to_frame()

,Spearman correlation with next month's log deaths
zero_streak,-0.522
outside_core_share_rm3,-0.106
momentum_3v12,0.114
hhi_rm3,0.157
civ_share_rm3,0.243
uncert_index_rm3,0.595
y_seasonal,0.760
n_admin1_rm3,0.813
log_sources_rm3,0.853
log_events_rm3,0.865


In [5]:
import plotly.graph_objects as go
c = sup[num].corr(method="spearman")
go.Figure(go.Heatmap(z=c.values, x=c.columns, y=c.index, zmid=0, colorscale="RdBu_r")).update_layout(
    title="Feature correlation (Spearman)", height=650)

### Leakage checks
* Target month > origin month for every row (asserted below).
* Annual context for year *Y* comes from *Y − 1*.
* 2026 rows are never in a training set for evaluated models (enforced in `src/models.py`).

In [6]:
sup_h = make_supervised(cm, ["conflict_case"], horizons=range(1, 10))
assert (sup_h.target_month > sup_h.month).all()
ctx = features.build_country_year_context()
print("context years available:", ctx.year.min(), "-", ctx.year.max(), "| used with +1 year lag")
print("rows with 2026 targets:", int((sup_h.target_month >= "2026-01-01").sum()), "(test only)")

context years available: 2014 - 2025 | used with +1 year lag
rows with 2026 targets: 504 (test only)
